In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True


# create_agent() -> CompiledStateGraph
```python
create_agent(
    model: str | BaseChatModel,  # LLM model name/string OR chat model object
    tools: Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None = None,  # Tools agent can use
    *,
    system_prompt: str | SystemMessage | None = None,  # Main instruction/personality
    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),  # Extra logic around agent steps
    response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,  # Structured output format
    state_schema: type[AgentState[ResponseT]] | None = None,  # Custom agent state schema
    context_schema: type[ContextT] | None = None,  # Runtime context schema
    checkpointer: Checkpointer | None = None,  # Saves/restores agent state
    store: BaseStore | None = None,  # Long-term memory/storage
    interrupt_before: list[str] | None = None,  # Pause before specific nodes
    interrupt_after: list[str] | None = None,  # Pause after specific nodes
    debug: bool = False,  # Show debug details
    name: str | None = None,  # Optional agent/graph name
    cache: BaseCache[Any] | None = None,  # Cache responses/results
    transformers: Sequence[TransformerFactory] | None = None  # Modify/optimize graph/model behavior
) -> CompiledStateGraph[AgentState[ResponseT], ContextT, InputAgentState, OutputAgentState[ResponseT]]  # Returns compiled LangGraph agent

```
Creates an agent graph that calls tools in a loop until a stopping condition is met.

# Parameters

## Required

- `model`
  - Type: `str | BaseChatModel`
  - Default: Required
  - Use: Language model for the agent.

## Optional

- `tools`
  - Type: `Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None`
  - Default: `None`
  - Use: Tools available to the agent.

- `system_prompt`
  - Type: `str | SystemMessage | None`
  - Default: `None`
  - Use: System instruction for the agent. It is added to the beginning of the message list when calling the model

- `middleware`
  - Type: `Sequence[AgentMiddleware[StateT_co, ContextT]]`
  - Default: `()`
  - Use: Controls or modifies agent behavior at various stage.

- `response_format`
  - Type: `ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None`
  - Default: `None`
  - Use: Structured output format.

- `state_schema`
  - Type: `type[AgentState[ResponseT]] | None`
  - Default: `None`
  - Use: custom state schema that extends `AgentState`. If provided, it becomes the base state schema used by the agent, allowing custom state fields without creating custom middleware.

- `context_schema`
  - Type: `type[ContextT] | None`
  - Default: `None`
  - Use: defines the structure of extra runtime information such as api client, db connection, permission etc passed to the agent through context.

- `checkpointer`
  - Type: `Checkpointer | None`
  - Default: `None`
  - Use: Saves state for one conversation thread.

- `store`
  - Type: `BaseStore | None`
  - Default: `None`
  - Use: Saves data across threads or users.

- `interrupt_before`
  - Type: `list[str] | None`
  - Default: `None`
  - Use: Pauses before selected nodes. Useful if you want to add a user confirmation or other interrupt before taking an action.

- `interrupt_after`
  - Type: `list[str] | None`
  - Default: `None`
  - Use: Pauses after selected nodes. Useful if you want to return directly or run additional processing on an output.

- `debug`
  - Type: `bool`
  - Default: `False`
  - Use: Enables execution logs.

- `name`
  - Type: `str | None`
  - Default: `None`
  - Use: Names the compiled graph.

- `cache`
  - Type: `BaseCache[Any] | None`
  - Default: `None`
  - Use: Caches graph execution.

- `transformers`
  - Type: `Sequence[TransformerFactory] | None`
  - Default: `None`
  - Use: Adds stream transformers.


# Returns
```python
CompiledStateGraph[AgentState[ResponseT],ContextT,_InputAgentState,_OutputAgentState[ResponseT]]
```
- AgentState -> internal full graph state
- InputAgentState -> input state shape accepted by the graph
- OutputAgentState -> output state shape returned by the graph
- ContextT -> runtime context type

# create_agent()

In [5]:
from typing import Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain.agents import create_agent, AgentState
from langchain.tools import tool
from langchain_core.messages import SystemMessage
from langchain_core.caches import InMemoryCache

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


# 1. Non-empty tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


# 2. Non-empty structured response schema
class FinalAnswer(BaseModel):
    answer: str = Field(description="Final answer")
    explanation: str = Field(description="Short explanation")


# 3. Non-empty custom state schema
class CustomState(AgentState):
    user_name: str
    task_type: str


# 4. Non-empty runtime context schema
class RuntimeContext(TypedDict):
    session_id: str
    user_level: str


# 5. Non-empty memory/storage/cache objects
checkpointer = InMemorySaver()
store = InMemoryStore()
cache = InMemoryCache()


# 6. create_agent call with all arguments non-empty
agent:CompiledStateGraph = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[multiply],

    system_prompt=SystemMessage(content="You are a helpful assistant. Use tools when needed."),

    middleware=(),  # keep empty unless you have actual middleware

    response_format=FinalAnswer,

    state_schema=CustomState,
    context_schema=RuntimeContext,

    checkpointer=checkpointer,
    store=store,

    interrupt_before=["tools"],
    interrupt_after=["model"],

    debug=True,
    name="math_agent",

    cache=cache,

    transformers=None,  # keep None unless you have actual TransformerFactory
)

# CompiledStateGraph
## Base Classes
- Pregel[StateT, ContextT, InputT, OutputT]
- Generic[StateT, ContextT, InputT, OutputT]

## Inherited From `Pregel`

### Attributes
* `nodes`:
  * Type: `dict[str, PregelNode]`
  * Stores all the nodes present inside the graph.
  * Each node represents one executable step in the workflow.

* `channels`:
  * Type: `dict[str, BaseChannel | ManagedValueSpec]`
  * Stores the communication channels used by the graph.
  * Channels help pass values/state between nodes.

* `stream_mode`:
  * Type: `StreamMode`
  * Defines how output should be streamed.
  * By default, it usually streams in `"values"` mode.

* `stream_eager`:
  * Type: `bool`
  * Controls whether stream events should be emitted eagerly.
  * When enabled, events are pushed as soon as they are available.

* `output_channels`:
  * Type: `str | Sequence[str]`
  * Defines which channel or channels are considered final output channels.

* `stream_channels`:
  * Type: `str | Sequence[str] | None`
  * Defines which channels should be streamed.
  * By default, it streams all non-reserved channels.

* `interrupt_after_nodes`:
  * Type: `All | Sequence[str]`
  * Specifies nodes after which graph execution should pause.
  * Useful for debugging or human-in-the-loop workflows.

* `interrupt_before_nodes`:
  * Type: `All | Sequence[str]`
  * Specifies nodes before which graph execution should pause.
  * Useful when you want to inspect or modify state before a node runs.

* `input_channels`:
  * Type: `str | Sequence[str]`
  * Defines which channels receive input when the graph starts.

* `step_timeout`:
  * Type: `float | None`
  * Sets the maximum time allowed for a graph step to complete.
  * If set to `None`, there is no fixed timeout.

* `debug`:
  * Type: `bool`
  * Controls whether debug information is printed during graph execution.

* `checkpointer`:
  * Type: `Checkpointer`
  * Used to save and load graph state.
  * Important for persistence, memory, and resuming graph execution.

* `store`:
  * Type: `BaseStore | None`
  * Represents the memory store used by the graph.
  * Can be used for shared values.

* `cache`:
  * Type: `BaseCache | None`
  * Used to store cached node results.
  * Helps avoid repeated computation.

* `retry_policy`:
  * Type: `Sequence[RetryPolicy]`
  * Defines retry behavior when graph tasks fail.
  * If empty, retries are disabled.

* `cache_policy`:
  * Type: `CachePolicy | None`
  * Defines caching behavior for graph nodes.
  * Individual nodes can override this policy.

* `context_schema`:
  * Type: `type[ContextT] | None`
  * Specifies the schema for the context object passed to the workflow.
  * Context is extra runtime information separate from the main state.

* `config`:
  * Type: `RunnableConfig | None`
  * Stores runtime configuration for the graph.

* `name`:
  * Type: `str`
  * Stores the name of the graph.

* `trigger_to_nodes`:
  * Type: `Mapping[str, Sequence[str]]`
  * Maps triggers to the nodes that should run when those triggers occur.

* `node_error_handler_map`:
  * Type: `Mapping[str, str]`
  * Maps nodes to their error handlers.
  * Useful for handling failures inside specific nodes.

* `stream_transformers`:
  * Type: `tuple[Callable[[tuple[str, ...]], Any], ...]`
  * Stores transformer functions used while streaming graph output.

* `InputType`:
  * Type: `Any`
  * Represents the expected input type of the graph.

* `OutputType`:
  * Type: `Any`
  * Represents the expected output type of the graph.

* `stream_channels_list`:
  * Type: `Sequence[str]`
  * Stores the list of channels used for streaming.

* `stream_channels_asis`:
  * Type: `str | Sequence[str]`
  * Stores the stream channel configuration in its original form.

### Methods
* `get_graph()`:
  * Returns a drawable representation of the computation graph.
  * Commonly used to visualize the graph structure.

* `aget_graph()`:
  * Asynchronous version of `get_graph()`.
  * Used when working inside async code.

* `copy()`:
  * Creates a copy of the `Pregel` object.

* `with_config()`:
  * Creates a copy of the graph with updated configuration.
  * Useful for changing runtime settings like recursion limits.

* `validate()`:
  * Validates the graph structure.
  * Checks whether the graph is properly configured.

* `config_schema()`:
  * Returns the schema for the graph configuration.

* `get_config_jsonschema()`:
  * Returns the configuration schema in JSON schema format.

* `get_context_jsonschema()`:
  * Returns the context schema in JSON schema format.

* `get_input_schema()`:
  * Returns the input schema of the graph.

* `get_output_schema()`:
  * Returns the output schema of the graph.

* `get_subgraphs()`:
  * Gets the subgraphs inside the main graph.
  * Useful when the graph contains nested graphs.

* `aget_subgraphs()`:
  * Asynchronous version of `get_subgraphs()`.

* `get_state()`:
  * Gets the current state of the graph.
  * Usually requires a checkpointer.

* `aget_state()`:
  * Asynchronous version of `get_state()`.

* `get_state_history()`:
  * Gets the history of graph states.
  * Useful for debugging and inspecting previous execution steps.

* `aget_state_history()`:
  * Asynchronously gets the history of graph states.

* `bulk_update_state()`:

  * Applies multiple updates to the graph state in bulk.
  * Requires a checkpointer to be set.

* `abulk_update_state()`:

  * Asynchronous version of `bulk_update_state()`.

* `update_state()`:

  * Updates the graph state with given values.
  * It behaves as if those values came from a graph node.

* `aupdate_state()`:
  * Asynchronous version of `update_state()`.

* `stream()`:
  * Streams graph steps for a single input.
  * Useful when you want to see intermediate outputs.

* `astream()`:
  * Asynchronously streams graph steps for a single input.

* `stream_events()`:
  * Streams detailed events from the graph execution.
  * Gives more detailed execution information than normal streaming.

* `astream_events()`:
  * Asynchronous version of `stream_events()`.

* `invoke()`:
  * Runs the graph with a single input and configuration.
  * This is one of the most commonly used execution methods.

* `ainvoke()`:
  * Asynchronous version of `invoke()`.

* `clear_cache()`:
  * Clears cached results for the given nodes.

* `aclear_cache()`:
  * Asynchronous version of `clear_cache()`.

---

## Inherited From `PregelProtocol`
### Methods
* `with_config()`:
  * Creates a copy of the graph with updated configuration.

* `get_graph()`:
  * Returns the drawable graph representation.

* `aget_graph()`:
  * Asynchronous version of `get_graph()`.

* `get_state()`:
  * Returns the current graph state.

* `aget_state()`:
  * Asynchronous version of `get_state()`.

* `get_state_history()`:
  * Returns the history of graph states.

* `aget_state_history()`:
  * Asynchronous version of `get_state_history()`.

* `bulk_update_state()`:
  * Applies multiple state updates in bulk.
  * Requires a checkpointer.

* `abulk_update_state()`:
  * Asynchronous version of `bulk_update_state()`.

* `update_state()`:
  * Updates the graph state manually.

* `aupdate_state()`:
  * Asynchronous version of `update_state()`.

* `stream()`:
  * Streams graph output step by step.

* `astream()`:
  * Asynchronous version of `stream()`.

* `invoke()`:
  * Runs the graph for a single input.

* `ainvoke()`:
  * Asynchronous version of `invoke()`.


In [ ]:
import json
#methods
from IPython.display import Image, display
print("Computation graph:")
display(Image(agent.get_graph().draw_mermaid_png()))
print("Asynchronous computation graph:")
display(Image(await agent.aget_graph().draw_mermaid_png()))

print("Config JSON Schema:")
print(json.dumps(agent.get_config_jsonschema(), indent=2))
print("Context JSON Schema:")
print(json.dumps(agent.get_context_jsonschema(), indent=2))

print("input JSON Schema:")
print(agent.get_input_schema())
print("output JSON Schema:")
print(agent.get_output_schema())

print("Synchronous subgraphs:")
for subgraph in agent.get_subgraphs():
    print(subgraph.name)
    display(Image(subgraph.draw_mermaid_png()))
print("Asynchronous subgraphs:")
for subgraph in await agent.aget_subgraphs():
    print(subgraph.name)
    display(Image(subgraph.draw_mermaid_png()))

print("Synchronous State")
print(agent.get_state())
print("Asynchronous State")
print(await agent.aget_state())

print("History of state of graph:")
for state:StateSnapshot in agent.get_state_history():
    print(state)
print("Asynchronous history of state of graph:")
for state:StateSnapshot in await agent.aget_state_history():
    print(state)

# agent.bulk_update_state()
# agent.abulk_update_state()
# agent.update_staete()
# agent.aupdate_state()

# print("Synchronous Stream:")
# for key:str, value:Any in Pregel(agent).stream():
#     print(f"key: {key}, value: {value}")
# print("Asynchronous Stream:")
# for key:str, value:Any in await Pregel(agent).astream():
#     print(f"key: {key}, value: {value}")

# print("Stream Event:")
# print(Pregel(agent).stream_event())
# print("Asynchronous Stream Event:")
# print(await Pregel(agent).astream_event())

# print("Invoke:")
# for key:str, value:Any in Pregel(agent).invoke():
#     print(f"key: {key}, value: {value}")
# print("Asynchronous Invoke:")
# for key:str, value:Any in await Pregel(agent).ainvoke():
#     print(f"key: {key}, value: {value}")

# agent.clear_cache()
# await agent.aclear_cache()

SyntaxError: invalid syntax (2422149265.py, line 34)

## Inherited From Runnable

`CompiledStateGraph` also inherits from LangChain's Runnable.
Runnable is a common LangChain interface for anything that can take input and return output.
This means it can be used like other LangChain runnable objects, such as models, chains, tools, and pipelines.

### Attributes
* `name`:
  * Stores the name of the runnable.

* `InputType`:
  * Represents the expected input type.

* `OutputType`:
  * Represents the expected output type.

* `input_schema`:
  * Stores the schema of the input accepted by the runnable.

* `output_schema`:
  * Stores the schema of the output produced by the runnable.

* `config_specs`:
  * Stores configuration specifications supported by the runnable.

### Methods

* `get_name()`:
  * Returns the name of the runnable.

* `get_input_schema()`:
  * Returns the input schema.

* `get_output_schema()`:
  * Returns the output schema.

* `config_schema()`:
  * Returns the configuration schema.

* `get_config_jsonschema()`:
  * Returns the configuration schema in JSON schema format.

* `get_graph()`:
  * Returns the internal graph representation.

* `get_prompts()`:
  * Returns prompts used inside the runnable, if available.

* `pipe()`:
  * Connects this runnable with another runnable.
  * Useful for building chains.

* `pick()`:
  * Selects specific keys from output dictonary of runnable. it is useful when chain return many fields, but you only want one or few fields

* `assign()`:
  * Adds new fields to the dictonary output of runnable.

* `invoke()`:
  * Runs the runnable with a single input.

* `ainvoke()`:
  * Asynchronous version of `invoke()`.

* `batch()`:
  * Runs the runnable on multiple inputs together parallely i.e. invoke() multiple time parallely.

* `batch_as_completed()`:
  * Runs multiple inputs and returns results as they finish. i.e. return result of each as soon as finishes

* `abatch()`:
  * Asynchronous version of `batch()`.

* `abatch_as_completed()`:
  * Asynchronous version of `batch_as_completed()`.

* `stream()`:
  * Streams runnable output i.e. it return output chunks one by one

* `astream()`:
  * Asynchronous version of `stream()`.

* `astream_log()`:
  * Streams logs asynchronously.

* `astream_events()`:
  * Streams execution events asynchronously.

* `stream_events()`:
  * Streams detailed execution events from a `Runnable`.
  * Useful for debugging, tracing, and understanding internal execution flow.
  * Syntax:
    ```python
    stream_events(
        self,
        input: Any,                                  # Input passed to the Runnable
        config: RunnableConfig | None = None,        # Optional runtime config for tracing/debugging
        *,
        version: Literal["v1", "v2", "v3"] = "v2", # Event schema version. For sync usage, version="v3" is generally used when the subclass supports the v3 streaming protocol.
        include_names: Sequence[str] | None = None,  # Include events from matching Runnable names
        include_types: Sequence[str] | None = None,  # Include events from matching Runnable types
        include_tags: Sequence[str] | None = None,   # Include events from matching tags
        exclude_names: Sequence[str] | None = None,  # Exclude events from matching Runnable names
        exclude_types: Sequence[str] | None = None,  # Exclude events from matching Runnable types
        exclude_tags: Sequence[str] | None = None,   # Exclude events from matching tags
        **kwargs: Any = {}                           # Extra arguments passed to the Runnable
    ) -> Iterator[StreamEvent] | Iterator[Any]

* `transform()`:
  * Transforms a stream/iterator of inputs into a stream/iterator of outputs. i.e. input chunks → output chunks
  * Useful when a `Runnable` needs to process input chunks and produce output chunks.
  * `stream()` works on one complete input, while `transform()` works on an iterator of inputs.
  * Syntax:
    ```python
    transform(
        self,
        input: Iterator[Input],                      # Iterator/stream of inputs passed to the Runnable
        config: RunnableConfig | None = None,        # Optional runtime config for tracing/debugging/execution
        **kwargs: Any | None = {}                    # Extra arguments passed to the Runnable
    ) -> Iterator[Output]
    ```

* `atransform()`:
  * Asynchronous version of `transform()`.

* `bind()`:
  * Binds fixed arguments to the runnable.
  * Useful when you want to permanently attach certain parameters.

* `with_config()`:
  * Creates a runnable with updated configuration.

* `with_listeners()`:
  * Binds lifecycle listeners to a `Runnable` and returns a new `Runnable`.
  * Useful when you want to run custom code when the runnable starts, ends, or fails.
  * Syntax:
    ```python
    with_listeners(
        self,
        *,
        on_start: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None,  # Called before Runnable starts
        on_end: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None,      # Called after Runnable finishes
        on_error: Callable[[Run], None] | Callable[[Run, RunnableConfig], None] | None = None     # Called if Runnable raises an error
    ) -> Runnable[Input, Output]
    ```
  * The `Run` object contains details about the execution, such as:
    ```text
    run id, run type, input, output, error, start_time, end_time, tags, metadata
    ```


* `with_alisteners()`:
  * Asynchronous version of `with_listeners()`.

* `with_types()`:
  * Specifies or overrides input and output types.

* `with_retry()`:
  * Creates a new `Runnable` that retries the original runnable if it fails.
  * Useful when the runnable may fail temporarily, such as during API calls, model calls, network calls, or unstable functions.
  * Syntax:
    ```python
    with_retry(
        self,
        *,
        retry_if_exception_type: tuple[type[BaseException], ...] = (Exception,),  # Exception types on which retry should happen
        wait_exponential_jitter: bool = True,                                     # Whether to add random delay/jitter between retries
        exponential_jitter_params: ExponentialJitterParams | None = None,         # Custom exponential jitter settings
        stop_after_attempt: int = 3                                               # Maximum number of attempts before giving up
    ) -> Runnable[Input, Output]
    ```
* `map()`:
  * Applies the runnable to each item in a list.

* `with_fallbacks()`:
  * Adds fallback runnables to a `Runnable` and returns a new `Runnable`.
  * The original runnable is tried first. If it fails, fallback runnables are tried one by one in order.
  * Syntax:
    ```python
    with_fallbacks(
        self,
        fallbacks: Sequence[Runnable[Input, Output]],                         # Runnables to try if the original Runnable fails
        *,
        exceptions_to_handle: tuple[type[BaseException], ...] = (Exception,), # Exception types that should trigger fallback
        exception_key: str | None = None                                      # Key used to pass exception info to fallback input
    ) -> RunnableWithFallbacksT[Input, Output]
    ```

* `as_tool()`:
  * Converts a `Runnable` into a agent callable `BaseTool`.
  * It can infer the tool schema automatically, or you can provide the schema manually.
  * Syntax:
    ```python
    as_tool(
        self,
        args_schema: type[BaseModel] | None = None,  # Pydantic schema for tool arguments
        *,
        name: str | None = None,                     # Name of the tool
        description: str | None = None,              # Description of what the tool does
        arg_types: dict[str, type] | None = None     # Simple argument names and their types
    ) -> BaseTool
    ```



In [ ]:
from langchain_core.runnables import Runnable, RunnableLambda, RunnableMap
from langchain_core.tracers.schemas import Run

from IPython.display import Image, display
import json

print("name:",Runnable.get_name(agent))
print("Input schema:",Runnable.get_input_schema(agent))
print("Output schema:",Runnable.get_output_schema(agent))

print("Config JSON Schema:")
for key, value in Runnable.get_config_jsonschema(agent).items():
    print(f"key: {key}, value: {value}")

print("Runnable graph:")
print(json.dumps(Runnable.get_graph(agent).to_json(), indent=2))

print("list of prompts used by This runnable")
for basePromptTemplate in Runnable.get_prompts(agent):
    print(str(basePromptTemplate))

print("Pipe:")
runnable1=RunnableLambda(lambda x: x+1)
runnable2=RunnableLambda(lambda x: x*2)
sequence=runnable1.pipe(runnable2) # equivalently sequence = runnable1 | runnable2 (or) sequence = RunnableSequence(first=runnable1, last=runnable2)
print(sequence.invoke(3))  # Output: 8 : execute one after another
print(sequence.batch([1,2,3])) #output [8, 10, 12]: execute one after another for each input in the list

print("Pick:")
chain = RunnableMap(
    string=RunnableLambda(str),
    json=RunnableLambda(json.loads),
    bytes=RunnableLambda(lambda x:bytes(x, 'utf-8'))
)
print(chain.invoke("[1,2,3]")) # Output: {'string': '[1,2,3]', 'json': [1, 2, 3], 'bytes': b'[1,2,3]'}
print(chain.pick(["json","string"]).invoke("[1,2,4]")) #Output: {'json': [1, 2, 4], 'string': '[1,2,4]'}
print(chain.pick("json").invoke("[1,2,3]")) # Output: [1, 2, 3]

print("assign:")
chain_with_assign=RunnableLambda(lambda x:{
    "name":x,
    "length":len(x)
}).assign(upper_name=lambda output: output["name"].upper())
print(chain_with_assign.invoke("hello")) #Output: {'name': 'hello', 'length': 5, 'upper_name': 'HELLO'}

print("batch:")
print(RunnableLambda(lambda x:10/x).batch(
    [5,2,0,3], # many input in parallel
    return_exceptions=True, #Optional,,true: raise error as output, false: raise error
    max_concurrency=2 #optional, maximum number of concurrent executions
    )
) #Output: [2.0, 5.0, ZeroDivisionError('division by zero'), 3.3333333333333335]

print("batch_as_completed:") # return output as soon as it completes
for result in RunnableLambda(lambda x:10/x).batch_as_completed(
    [5,2,0,3], # many input in parallel
    return_exceptions=True, #Optional,,true: raise error as output, false: raise error
    max_concurrency=2 #optional, maximum number of concurrent executions
    ):
    print(result) #Output: 2.0, 5.0, ZeroDivisionError('division by zero'), 3.3333333333333335

print("stream:")
for chunk in RunnableLambda(lambda x:x+1).stream(10):
    print(chunk) #Output: 11

# print("stream events")
# for event in agent.stream_events():
#     print(event) #Output: StreamEvent(type='start', data=None), StreamEvent(type='data', data=StateSnapshot(state={'user_name': 'Alice', 'task_type': 'math'})), StreamEvent(type='end', data=None)

print("transform:")
for output in RunnableLambda(lambda x:x.upper()).transform(iter(["hello","world"])):
    print(output) #Output: HELLO, WORLD

print("with listener:")
chain_with_listener=RunnableLambda(lambda x:x+1).with_listener(
    on_start=lambda run: print(f"Starting run with input: {run.inputs}"),
    on_end=lambda run: print(f"Ending run with output: {run.outputs}")
)
print(chain_with_listener.invoke(5)) #Output: Starting run with input: 5, Ending run with output: 6, 6

print("with type:")
runnable_with_type=RunnableLambda(lambda x:x+1).with_type(int)
print(runnable_with_type.invoke(5)) #Output: 6

# print("with_retry:")
# num=-1
# RunnableLambda(lambda x: raise ValueError("Negative") if x < 0 else x=x+1).with_retry(
#     stop_after_attempts=3, # maximum number of attempts
#     retry_if_exception=lambda e: isinstance(e, ValueError) and str(e) == "Negative" # retry only if the exception is ValueError with message "Negative"
# ).invoke(num) #Output: ValueError: Negative after 3 attempts

print("map:")
print(RunnableLambda(lambda x:x+1).map([1,2,3])) #Output: [2, 3, 4]

print("with_fallbacks:")
def primary_model(x):
    raise ValueError("Primary failed")
def backup_model(x):
    return f"Backup answer for: {x}"

primary = RunnableLambda(primary_model)
backup = RunnableLambda(backup_model)

chain = primary.with_fallbacks([backup])

result = chain.invoke("Hello")
print(result)

print("as tool")
runnable = RunnableLambda(lambda x: str(x["a"] * max(x["b"])))
# Convert Runnable into Tool
tool = runnable.as_tool(
    name="multiply_with_max",
    description="Multiplies a number with the maximum value from a list.",
    arg_types={"a": int,"b": list[int]}
)
# Use tool
result = tool.invoke({"a": 3,"b": [1, 2, 5]})
print(result)




name: math_agent
Input schema: <class 'langgraph.graph.state.math_agent_input'>
Output schema: <class 'langchain_core.utils.pydantic.math_agent_output'>
Config JSON Schema:
key: $defs, value: {'RuntimeContext': {'properties': {'session_id': {'title': 'Session Id', 'type': 'string'}, 'user_level': {'title': 'User Level', 'type': 'string'}}, 'required': ['session_id', 'user_level'], 'title': 'RuntimeContext', 'type': 'object'}}
key: properties, value: {'configurable': {'$ref': '#/$defs/RuntimeContext', 'default': None}}
key: title, value: math_agent_config
key: type, value: object
Runnable graph:
{
  "nodes": [
    {
      "id": 0,
      "type": "schema",
      "data": "math_agent_input"
    },
    {
      "id": 1,
      "type": "runnable",
      "data": {
        "id": [
          "langgraph",
          "graph",
          "state",
          "CompiledStateGraph"
        ],
        "name": "math_agent"
      }
    },
    {
      "id": 2,
      "type": "schema",
      "data": "math_agent_o

In [ ]:
import json
#methods
print("Input JSON Schema:")
print(json.dumps(agent.get_input_jsonschema(), indent=2))

print("Output JSON Schema:")
print(json.dumps(agent.get_output_jsonschema(), indent=2))



#agent.attach_node()
#agent.attach_edge()
#agent.attach_branch()




#Runnable
